# Planners-5b — Admissibilité de la relaxation sans-delete (companion formel natif)

Ce notebook est le **companion formel** du lake [`planning_lean`](../planning_lean/),
dont la lib `Planning` prouve l'**admissibilité de la relaxation** (heuristique $h^+$) :
la relaxation *sans-delete* ne surestime jamais le coût réel ($h^+ \le h^*$) — un
résultat central de la planification classique (Pearl 1984, Bonet & Geffner 2001),
avec **zéro `sorry`**.

L'idée : dans la **transition relaxée** `stepR`, on ignore les effets de *deletion*
des actions STRIPS. Comme `step a s ⊆ stepR a s` (on enlève moins), tout plan réel est
atteignable en relaxation, d'où $h^+ \le h^*$.

## Convention de vérification — `#check` *natif* dans le kernel Lean

Ce notebook est un **notebook Lean natif** (kernel `lean4-wsl`) : il `import`e les
modules du lake directement et le compilateur Lean rend les signatures **dans le
notebook**. C'est rendu possible par l'UNLOCK (patch `lean4_jupyter` + jonction
Mathlib #2611).

> ⚠️ À l'exécution, la première cellule (`import`) peut prendre **~3 min** :
> le kernel charge les oleans Mathlib via la jonction NTFS. Les suivantes sont
> instantanées.

## 1. Import des modules du lake `planning_lean`

Le lake est structuré en 3 modules (`Strips`, `Relaxation`, `Admissibility`) sous le
namespace `PlanningLean`. On importe les 3 modules (la config `submodules` du lake ne
build pas l'umbrella racine — on cible directement les modules qui portent les
définitions).

In [1]:
import Planning.Strips
import Planning.Relaxation
import Planning.Admissibility
open PlanningLean

import Planning.Strips
import Planning.Relaxation
import Planning.Admissibility
open PlanningLean
--% env 0

Raw input:
{"cmd": "import Planning.Strips\nimport Planning.Relaxation\nimport Planning.Admissibility\nopen PlanningLean"}
Raw output:
{"env": 0}

## 2. Preuve sans `sorry` — `#print axioms`

Le théorème phare `relaxed_plan_admissible` (admissibilité $h^+ \le h^*$) ne dépend que
des **3 axiomes standards de Lean** (`propext`, `Classical.choice`, `Quot.sound`) — pas
de `sorryAx` — ce qui prouve que la preuve est **complète** (0 sorry).

In [2]:
#print axioms PlanningLean.relaxed_plan_admissible

#print axioms PlanningLean.relaxed_plan_admissible
──────▶  'PlanningLean.relaxed_plan_admissible' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 1

Raw input:
{"cmd": "#print axioms PlanningLean.relaxed_plan_admissible", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'PlanningLean.relaxed_plan_admissible' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 1}

## 3. Le modèle STRIPS — fluents, états, actions, transitions

Le module `Planning/Strips.lean` pose le cadre de la planification STRIPS :

- `State F = Finset F` — un état = l'ensemble des fluents actuellement vrais.
- `Action F` — une action : préconditions `pre`, effets additifs `add`, deletions `del`.
- `step a s = (s \ a.del) ∪ a.add` — la **transition réelle** (on retire les deletions).
- `stepR a s = s ∪ a.add` — la **transition relaxée** (on ignore les deletions).

Le lemme clé `step_subset_stepR` : `step a s ⊆ stepR a s` car `s \ a.del ⊆ s`.

In [3]:
#check State
#check Action
#check step
#check stepR
#check step_subset_stepR

#check State
──────▶  PlanningLean.State (F : Type) : Type
#check Action
──────▶  PlanningLean.Action (F : Type) : Type
#check step
──────▶  PlanningLean.step {F : Type} [DecidableEq F] (a : PlanningLean.Action F) (s : State F) : State F
#check stepR
──────▶  PlanningLean.stepR {F : Type} [DecidableEq F] (a : PlanningLean.Action F) (s : State F) : State F
#check step_subset_stepR
──────▶  PlanningLean.step_subset_stepR {F : Type} [DecidableEq F] (a : PlanningLean.Action F) (s : State F) :
  step a s ⊆ stepR a s
--% env 2

Raw input:
{"cmd": "#check State\n#check Action\n#check step\n#check stepR\n#check step_subset_stepR", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "PlanningLean.State (F : Type) : Type"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "PlanningLean.Action (F : Type) : Type"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PlanningLean.step {F : Type} [DecidableEq F] (a : PlanningLean.Action F) (s : State F) : State F"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "PlanningLean.stepR {F : Type} [DecidableEq F] (a : PlanningLean.Action F) (s : State F) : State F"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "PlanningLean.step_subset_stepR {F : Type} [DecidableEq F] (a : PlanningLean.Action F) (s : State F) :\n  step a s ⊆ stepR a s"}],
 "env": 2}

### Lecture : réel vs relaxé, au niveau d'une action

| Symbole Lean | Lecture |
|-------------|---------|
| `State F` | état = ensemble de fluents (`Finset F`) |
| `Action F` | action (pre/add/del) |
| `step a s` | transition réelle : `(s \ del) ∪ add` |
| `stepR a s` | transition relaxée : `s ∪ add` (ignore `del`) |
| `step_subset_stepR` | `step a s ⊆ stepR a s` (cœur de la relaxation) |

## 4. Exécution relaxée des plans — monotonie et lemme central

Le module `Planning/Relaxation.lean` étend la transition au plan entier :

- `run π s` / `runR π s` — exécution (réelle / relaxée) d'une séquence d'actions.
- `reaches π s g` / `reachesR π s g` — le plan atteint le but `g` depuis `s`.
- `runR_mono` — l'exécution relaxée est **monotone** en l'état initial (`s ⊆ s'` ⟹ `runR π s ⊆ runR π s'`), ce qui rend le calcul relaxé polynomial.
- `run_subset_runR` — le lemme central : toute exécution réelle est incluse dans la relaxation (`run π s ⊆ runR π s`), par induction sur la longueur du plan.

In [4]:
#check run
#check runR
#check reaches
#check reachesR
#check runR_mono
#check run_subset_runR

#check run
──────▶  PlanningLean.run {F : Type} [DecidableEq F] : List (PlanningLean.Action F) → State F → State F
#check runR
──────▶  PlanningLean.runR {F : Type} [DecidableEq F] : List (PlanningLean.Action F) → State F → State F
#check reaches
──────▶  PlanningLean.reaches {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F) : Prop
#check reachesR
──────▶  PlanningLean.reachesR {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F) : Prop
#check runR_mono
──────▶  PlanningLean.runR_mono {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) {s s' : State F} (h : s ⊆ s') :
  runR π s ⊆ runR π s'
#check run_subset_runR
──────▶  PlanningLean.run_subset_runR {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s : State F) :
  run π s ⊆ runR π s
--% env 3

Raw input:
{"cmd": "#check run\n#check runR\n#check reaches\n#check reachesR\n#check runR_mono\n#check run_subset_runR", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PlanningLean.run {F : Type} [DecidableEq F] : List (PlanningLean.Action F) → State F → State F"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PlanningLean.runR {F : Type} [DecidableEq F] : List (PlanningLean.Action F) → State F → State F"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PlanningLean.reaches {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F) : Prop"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "PlanningLean.reachesR {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F) : Prop"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "PlanningLean.runR_mono {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) {s s' : State F} (h : s ⊆ s') :\n  runR π s ⊆ runR π s'"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "PlanningLean.run_subset_runR {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s : State F) :\n  run π s ⊆ runR π s"}],
 "env": 3}

### Lecture : du pas local au plan entier

| Symbole Lean | Lecture |
|--------------|---------|
| `run π s` | exécution réelle du plan `π` depuis `s` |
| `runR π s` | exécution relaxée (sans-delete) |
| `reaches π s g` | `π` atteint `g` en exécution réelle |
| `runR_mono` | monotonie relaxée (calcul polynomial) |
| `run_subset_runR` | `run π s ⊆ runR π s` (induction sur `π`) |

## 5. Théorème phare : l'admissibilité de la relaxation ($h^+ \\le h^*$)

Le module `Planning/Admissibility.lean` prouve le résultat central : si un plan réel
`π` atteint le but `g` depuis `s`, alors le plan relaxé `π` atteint aussi `g` — en une
**transitivité** `run ⊆ runR` puis `goalSatisfied` préserve l'inclusion.

- `relaxed_plan_admissible` : `reaches π s g → reachesR π s g` (toute atteignabilité réelle est relaxée ⟹ $h^+ \\le h^*$).
- `relaxed_plan_witness` : variante existentielle (le plan relaxé est un témoin d'atteignabilité).

In [5]:
#check PlanningLean.relaxed_plan_admissible
#check PlanningLean.relaxed_plan_witness

#check PlanningLean.relaxed_plan_admissible
──────▶  PlanningLean.relaxed_plan_admissible {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F)
  (h : reaches π s g) : reachesR π s g
#check PlanningLean.relaxed_plan_witness
──────▶  PlanningLean.relaxed_plan_witness {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F)
  (h : reaches π s g) : reachesR π s g
--% env 4

Raw input:
{"cmd": "#check PlanningLean.relaxed_plan_admissible\n#check PlanningLean.relaxed_plan_witness", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PlanningLean.relaxed_plan_admissible {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F)\n  (h : reaches π s g) : reachesR π s g"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PlanningLean.relaxed_plan_witness {F : Type} [DecidableEq F] (π : List (PlanningLean.Action F)) (s g : State F)\n  (h : reaches π s g) : reachesR π s g"}],
 "env": 4}

### Lecture : $h^+ \\le h^*$, en une transitivité

| Théorème | Conclusion |
|----------|-----------|
| `relaxed_plan_admissible` | `reaches π s g → reachesR π s g` (h⁺ ≤ h\*) |
| `relaxed_plan_witness` | variante existentielle (témoin relaxé) |

La relaxation **ne surestime jamais le coût réel** : tout plan réalisable en réalité
l'est aussi en relaxation, donc l'heuristique $h^+$ (coût relaxé) minore $h^*$ (coût réel).

## 6. La chaîne causale complète

Les trois modules composent une chaîne unique, du modèle STRIPS à l'admissibilité :

1. **`Strips`** — `State`/`Action`, transitions `step`/`stepR`, `step ⊆ stepR`.
2. **`Relaxation`** — `run`/`runR` sur le plan, `runR_mono`, `run ⊆ runR` (induction).
3. **`Admissibility`** — `reaches → reachesR` (transitivité) ⟹ $h^+ \\le h^*$.

## 7. Exercices

### Exercice 1 — Transitions réelle/relaxée sur un mini-STRIPS (Python)

Ancrez l'intuition : un domaine à 2 actions (`load`, `unload`) et observez que la
transition relaxée est toujours un sur-ensemble de la transition réelle.

```python
# action = (pre, add, del)
load = (pre={'at-truck'}, add={'loaded'}, del=set())
unload = (pre={'loaded'}, add={'delivered'}, del={'loaded'})
def step(action, state):
    # TODO etudiant : transition reelle (s \ del) ∪ add
    return None  # TODO
def stepR(action, state):
    # TODO etudiant : transition relaxee s ∪ add
    return None  # TODO
```

### Exercice 2 — Monotonie de la transition relaxée

Prouvez en Lean que `stepR` est monotone : `s ⊆ s' → stepR a s ⊆ stepR a s'`.
(C'est un cas particulier de `stepR_mono` sur une seule action.)

```lean
-- TODO etudiant : formaliser la monotonie de stepR
-- theorem stepR_mono_single (a : Action F) {s s' : State F} (h : s ⊆ s') :
--     stepR a s ⊆ stepR a s' := by sorry
```

### Exercice 3 — La relaxation peut être strictement moins chère

Construisez un exemple où la relaxation permet un plan « impossible » en réalité
(les deletions bloquent). Montrez que `reachesR` est vrai mais `reaches` faux :
la relaxation est strictement plus optimiste ($h^+ < h^*$), sans jamais dépasser $h^*$.

```python
# TODO etudiant : domaine ou un plan relaxe atteint le but mais pas le plan reel
# (ex: action avec deletion qui bloque la re-application necessaire)
```

## Conclusion

Ce companion **natif** exhibe la preuve formelle 0-sorry de l'admissibilité de la
relaxation ($h^+ \\le h^*$) dans le kernel Lean lui-même : `#check` et `#print axioms`
rendent les signatures et les axiomes réels produits par le compilateur, sans
intermédiaire Python.

Le résultat phare `relaxed_plan_admissible` formalise la propriété qui rend les
heuristiques relaxées (h-add, h-max, h-FF) **admissibles** — un pilier de la
planification classique moderne.

**Jalon ouvert** : la consistance de la relaxation (existence d'un plan relaxé) et la
complexité algorithmique ne sont pas formalisées dans le lake ; la lib reste sorry-free.